In [1]:
# =============================================================================
# ANÁLISE DE CORRELAÇÃO / ASSOCIAÇÃO
# =============================================================================
# Bibliotecas necessárias:
#   pip install pandas numpy scipy
# =============================================================================

import pandas as pd
import numpy as np
from scipy.stats import spearmanr, pearsonr, chi2_contingency, f_oneway

C:\Users\gcabr\AppData\Local\Programs\Python\Python311\Lib\site-packages\scipy\__init__.py:169: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
# =============================================================================
# 1. CARREGAR A BASE DE DADOS
# =============================================================================
# --- Base de exemplo para demonstração ---
np.random.seed(42)
n = 200

df = pd.DataFrame({
    "idade": np.random.randint(18, 70, n),
    "renda": np.random.normal(5000, 1500, n).round(2),
    "nota": np.random.uniform(0, 10, n).round(1),
    "sexo": np.random.choice(["M", "F"], n),
    "escolaridade": np.random.choice(["Fundamental", "Médio", "Superior", "Pós"], n),
    "regiao": np.random.choice(["Norte", "Sul", "Leste", "Oeste"], n),
})

In [3]:
df.head()

,idade,renda,nota,sexo,escolaridade,regiao
0,56,5190.52,9.4,F,Médio,Leste
1,69,3125.83,0.4,M,Superior,Norte
2,46,7917.67,4.2,F,Médio,Oeste
3,32,4770.00,9.7,M,Fundamental,Leste
4,60,3639.52,5.5,M,Médio,Sul


In [4]:
# =============================================================================
# 2. ESPECIFICAR AS VARIÁVEIS QUANTITATIVAS E QUALITATIVAS
# =============================================================================
# PREENCHA AS DUAS LISTAS ABAIXO com os nomes exatos das colunas da sua base.
# Os nomes devem corresponder exatamente aos nomes das colunas do DataFrame.
#

variaveis_quanti = ["idade", "renda", "nota"]
variaveis_quali  = ["sexo", "escolaridade", "regiao"]

# --- Validação: verifica se todas as variáveis existem na base ---
todas_especificadas = variaveis_quanti + variaveis_quali
colunas_base = list(df.columns)

# Checa variáveis que o usuário digitou mas não existem na base
inexistentes = [v for v in todas_especificadas if v not in colunas_base]
if inexistentes:
    raise ValueError(
        f"   ERRO: As seguintes variáveis não existem na base: {inexistentes}\n"
        f"   Colunas disponíveis: {colunas_base}"
    )

# Checa variáveis duplicadas (na mesma lista ou entre listas)
duplicadas = [v for v in todas_especificadas if todas_especificadas.count(v) > 1]
if duplicadas:
    raise ValueError(
        f"   ERRO: Variáveis duplicadas encontradas: {list(set(duplicadas))}\n"
        f"   Cada variável deve aparecer em apenas uma lista."
    )

# Checa colunas da base que não foram classificadas (aviso, não erro)
nao_classificadas = [v for v in colunas_base if v not in todas_especificadas]
if nao_classificadas:
    print(f"\n  Colunas da base NÃO incluídas na análise: {nao_classificadas}")
    print("    (Se quiser incluí-las, adicione aos vetores acima)")

print("\n" + "=" * 70)
print("CLASSIFICAÇÃO DAS VARIÁVEIS (definida pelo usuário)")
print("=" * 70)
print(f"Quantitativas ({len(variaveis_quanti)}): {variaveis_quanti}")
print(f"Qualitativas  ({len(variaveis_quali)}):  {variaveis_quali}")

# --- Diagnóstico de valores faltantes ---
colunas_analisadas = variaveis_quanti + variaveis_quali
nulos = df[colunas_analisadas].isnull().sum()
if nulos.sum() > 0:
    print(f"\n VALORES FALTANTES ENCONTRADOS:")
    print(nulos[nulos > 0])
    print("(Serão removidos par a par em cada cálculo)\n")
else:
    print("\n Nenhum valor faltante nas variáveis selecionadas.\n")



CLASSIFICAÇÃO DAS VARIÁVEIS (definida pelo usuário)
Quantitativas (3): ['idade', 'renda', 'nota']
Qualitativas  (3):  ['sexo', 'escolaridade', 'regiao']

 Nenhum valor faltante nas variáveis selecionadas.



In [5]:
# =============================================================================
# 3. FUNÇÕES DE CLASSIFICAÇÃO DE INTENSIDADE
# =============================================================================

# --- Pearson / Spearman ---

def classificar_correlacao(valor):
    """Classifica |r| ou |ρ| segundo Cohen (1988)."""
    v = abs(valor)
    if v < 0.10:   return "Desprezível"
    elif v < 0.30: return "Fraca"
    elif v < 0.50: return "Moderada"
    elif v < 0.70: return "Forte"
    else:          return "Muito forte"


# --- Correlação de Distância (dCor) ---

# NOTA: os patamares de Cohen são usados aqui como referência APROXIMADA.
# O dCor possui viés positivo em amostras finitas, ou seja, mesmo variáveis
# independentes podem gerar dCor > 0 por acaso. Valores baixos (< 0.15)
# devem ser interpretados com cautela.

def classificar_dcor(valor):
    """Classifica dCor — patamares aproximados baseados em Cohen."""
    v = abs(valor)
    if v < 0.10:   return "Desprezível"
    elif v < 0.30: return "Fraca"
    elif v < 0.50: return "Moderada"
    elif v < 0.70: return "Forte"
    else:          return "Muito forte"


# --- V de Cramér---

def classificar_cramer(valor):
    """Classifica V de Cramér segundo Rea & Parker (1992)."""
    v = abs(valor)
    if v < 0.10:   return "Desprezível"
    elif v < 0.20: return "Fraca"
    elif v < 0.40: return "Moderada"
    elif v < 0.60: return "Relativamente forte"
    else:          return "Forte"


# --- Eta ---
# Referência: Cohen, J. (1988).
# Eta² (eta ao quadrado) é interpretado como proporção de variância explicada.

def classificar_eta(valor):
    """Classifica Eta (η) segundo Cohen (1988)."""
    v = abs(valor)
    if v < 0.10:   return "Desprezível"
    elif v < 0.30: return "Fraca"
    elif v < 0.50: return "Moderada"
    else:          return "Forte"


def classificar_eta2(valor):
    """Classifica Eta² (η²) como tamanho de efeito — Cohen (1988)."""
    v = abs(valor)
    if v < 0.01:   return "Desprezível"
    elif v < 0.06: return "Pequeno"
    elif v < 0.14: return "Médio"
    else:          return "Grande"

In [6]:
# =============================================================================
# 4. CORRELAÇÃO DE DISTÂNCIA (dCor) — IMPLEMENTAÇÃO
# =============================================================================
# A dCor captura QUALQUER tipo de dependência entre duas variáveis:
# linear, monotônica, curva em U, senoidal, circular, etc.
#
# Se dCor = 0, as variáveis são estatisticamente INDEPENDENTES
# (propriedade que Pearson e Spearman NÃO têm).
#
# Algoritmo:
#   1. Calcula a matriz de distâncias euclidianas para cada variável
#      (a[i,j] = |x_i - x_j| para todos os pares i,j)
#   2. Centraliza duplamente cada matriz: subtrai médias de linha,
#      médias de coluna, e soma a média geral
#   3. dCov²(X,Y) = média do produto elemento a elemento das matrizes
#   4. dCor(X,Y) = sqrt( dCov(X,Y) / sqrt(dCov(X,X) * dCov(Y,Y)) )
#
# Custo computacional: O(n²) em memória e tempo (matrizes n×n).
# Para bases grandes, usa amostragem aleatória automática.

def correlacao_distancia(x, y, max_amostras=5000):
    """
    Calcula a Correlação de Distância entre dois vetores numéricos.

    Parâmetros:
        x, y: arrays numéricos de mesmo tamanho
        max_amostras: para n > max_amostras, amostra aleatoriamente
                      para evitar estouro de memória (matrizes n×n)

    Retorna: float entre 0 e 1
    """
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    n = len(x)

    # Amostragem reprodutível para bases grandes
    if n > max_amostras:
        rng = np.random.RandomState(42)  # semente fixa → reprodutível
        idx = rng.choice(n, max_amostras, replace=False)
        x = x[idx]
        y = y[idx]
        n = max_amostras

    # Passo 1: Matrizes de distância
    a = np.abs(x[:, None] - x[None, :])
    b = np.abs(y[:, None] - y[None, :])

    # Passo 2: Dupla centralização
    A = a - a.mean(axis=1, keepdims=True) - a.mean(axis=0, keepdims=True) + a.mean()
    B = b - b.mean(axis=1, keepdims=True) - b.mean(axis=0, keepdims=True) + b.mean()

    # Passo 3: Covariância de distância ao quadrado
    dcov2_xy = (A * B).mean()
    dcov2_xx = (A * A).mean()
    dcov2_yy = (B * B).mean()

    # Passo 4: Correlação de distância
    # max(0) evita raiz de valor negativo por imprecisão numérica
    # Tolerância 1e-15 evita divisão por denominador quase-zero
    denominador = np.sqrt(dcov2_xx * dcov2_yy)
    if denominador < 1e-15:
        return 0.0
    return np.sqrt(max(dcov2_xy, 0.0) / denominador)

In [7]:
# =============================================================================
# 5. V DE CRAMÉR CORRIGIDO POR VIÉS — IMPLEMENTAÇÃO
# =============================================================================
# O V de Cramér padrão tem viés positivo: superestima a associação,
# especialmente com tabelas grandes ou amostras pequenas.
#
# A correção de Bergsma (2013) ajusta tanto o χ² quanto as dimensões
# efetivas da tabela para remover esse viés.
#
# Fórmula corrigida:
#   χ²_corr = max(0, χ² - (r-1)(c-1)/(n-1))
#   r_corr  = r - (r-1)²/(n-1)
#   c_corr  = c - (c-1)²/(n-1)
#   V_corr  = sqrt( χ²_corr / (n × (min(r_corr, c_corr) - 1)) )
#
# Referência: Bergsma, W. (2013). A bias-correction for Cramér's V and
# Tschuprow's T. Journal of the Korean Statistical Society, 42(3), 323-328.

def calcular_cramer_corrigido(tabela, chi2, n_obs):
    """
    Calcula o V de Cramér com correção de viés (Bergsma, 2013).

    Parâmetros:
        tabela: pd.DataFrame (tabela de contingência)
        chi2: float (valor do qui-quadrado já calculado)
        n_obs: int (número total de observações)

    Retorna: float entre 0 e 1
    """
    r_tab = tabela.shape[0]  # número de linhas
    c_tab = tabela.shape[1]  # número de colunas

    # Corrige o chi2 subtraindo o viés esperado
    chi2_corrigido = max(0, chi2 - (r_tab - 1) * (c_tab - 1) / (n_obs - 1))

    # Corrige as dimensões efetivas da tabela
    r_corrigido = r_tab - (r_tab - 1) ** 2 / (n_obs - 1)
    c_corrigido = c_tab - (c_tab - 1) ** 2 / (n_obs - 1)
    k_corrigido = min(r_corrigido, c_corrigido) - 1

    if k_corrigido > 0 and n_obs > 0:
        return np.sqrt(chi2_corrigido / (n_obs * k_corrigido))
    else:
        return 0.0

In [8]:
# =============================================================================
# 6. CÁLCULO DO ETA — FUNÇÃO AUXILIAR
# =============================================================================
# Eta (η) = razão de correlação. Mede quanto uma variável categórica
# "explica" a variação de uma variável numérica.
#
# η = sqrt( SQ_entre / SQ_total )
#   SQ_total  = variação total da numérica (desvios em relação à média geral)
#   SQ_entre  = variação ENTRE grupos (quanto as médias dos grupos diferem
#               da média geral, ponderado pelo tamanho de cada grupo)
#
# η² = proporção da variância explicada (ex: 0.15 → 15%)
#
# O p-valor vem do teste ANOVA one-way (F-test), que testa a hipótese
# nula de que as médias de todos os grupos são iguais.

def calcular_eta(df, var_qual, var_quant):
    """
    Calcula Eta, Eta² e p-valor (ANOVA) entre uma qualitativa e uma quantitativa.
    Remove NaN automaticamente.
    Remove grupos com menos de 2 observações (ANOVA exige variância intra-grupo).

    Retorna: (eta, eta2, p_valor_anova)
    """
    sub = df[[var_qual, var_quant]].dropna()

    media_geral = sub[var_quant].mean()
    sq_total = ((sub[var_quant] - media_geral) ** 2).sum()

    # SQ_entre: soma de (tamanho_grupo × (média_grupo - média_geral)²)
    sq_entre = sum(
        len(g) * (g[var_quant].mean() - media_geral) ** 2
        for _, g in sub.groupby(var_qual)
    )

    # Eta
    eta = np.sqrt(sq_entre / sq_total) if sq_total != 0 else 0.0
    eta2 = eta ** 2

    # p-valor via ANOVA one-way
    # Filtra grupos com menos de 2 obs (ANOVA precisa de variância intra-grupo)
    grupos = [g[var_quant].values for _, g in sub.groupby(var_qual)]
    grupos = [g for g in grupos if len(g) >= 2]

    if len(grupos) >= 2:
        _, p_anova = f_oneway(*grupos)
    else:
        p_anova = float("nan")

    return eta, eta2, p_anova

In [15]:
# =============================================================================
# 7. QUANTI x QUANTI — Pearson, Spearman e dCor
# =============================================================================
# Pearson (r):   mede correlação LINEAR. Varia de -1 a +1.
#                Sensível a outliers. Se a relação for curva, pode dar ~0.
#
# Spearman (ρ):  mede correlação MONOTÔNICA (baseada em postos/rankings).
#                Captura relações "sempre crescente" ou "sempre decrescente",
#                mesmo que não sejam em linha reta. Mais robusto a outliers.
#
# dCor:          mede QUALQUER tipo de dependência. Se dCor > 0 e Pearson ~0,
#                existe uma relação não-linear que Pearson não enxerga.
#
# Os p-valores de Pearson e Spearman testam H0: "não há correlação" (r = 0).
# Valores < 0.05 indicam que a correlação é estatisticamente significativa.

print("\n" + "=" * 70)
print("QUANTI x QUANTI — Pearson (r), Spearman (ρ), dCor")
print("=" * 70)

resultados_qq = []

for i in range(len(variaveis_quanti)):
    for j in range(i + 1, len(variaveis_quanti)):
        var1 = variaveis_quanti[i]
        var2 = variaveis_quanti[j]

        # Remove NaN par a par (garante consistência entre as 3 medidas)
        pares = df[[var1, var2]].dropna()

        # Pearson — linear
        r_pearson, p_pearson = pearsonr(pares[var1], pares[var2])

        # Spearman — monotônica
        r_spearman, p_spearman = spearmanr(pares[var1], pares[var2])

        # dCor — qualquer relação
        dcor = correlacao_distancia(pares[var1].values, pares[var2].values)

        resultados_qq.append({
            "Var 1": var1,
            "Var 2": var2,
            "Pearson (r)": round(r_pearson, 4),
            "p-valor Pearson": round(p_pearson, 4),
            "Classif. Pearson": classificar_correlacao(r_pearson),
            "Spearman (ρ)": round(r_spearman, 4),
            "p-valor Spearman": round(p_spearman, 4),
            "Classif. Spearman": classificar_correlacao(r_spearman),
            "dCor": round(dcor, 4),
            "Classif. dCor": classificar_dcor(dcor),
        })

if resultados_qq:
    df_qq = pd.DataFrame(resultados_qq)
else:
    print("Não há pares de variáveis quantitativas suficientes.")


QUANTI x QUANTI — Pearson (r), Spearman (ρ), dCor


In [16]:
df_qq.head(40)

,Var 1,Var 2,Pearson (r),p-valor Pearson,Classif. Pearson,Spearman (ρ),p-valor Spearman,Classif. Spearman,dCor,Classif. dCor
0,idade,renda,0.0992,0.1621,Desprezível,0.0850,0.2316,Desprezível,0.1102,Fraca
1,idade,nota,0.0681,0.3383,Desprezível,0.0638,0.3695,Desprezível,0.1025,Fraca
2,renda,nota,0.1552,0.0282,Fraca,0.1735,0.0140,Fraca,0.2021,Fraca


In [17]:
# =============================================================================
# 8. QUALI x QUALI — V de Cramér (corrigido por viés)
# =============================================================================
# O V de Cramér mede a associação entre duas variáveis categóricas.
# É derivado do teste Qui-Quadrado e normalizado para variar de 0 a 1.
#
# Esta versão usa a correção de viés de Bergsma (2013), que evita a
# superestimação do Cramér padrão em amostras pequenas ou tabelas grandes.
#
# O p-valor vem do teste Qui-Quadrado e testa H0: "as variáveis são
# independentes". Valores < 0.05 indicam associação significativa.
#
# AVISO: O teste Qui-Quadrado assume que todas as frequências esperadas
# sejam >= 5. Quando isso não ocorre, o p-valor pode ser impreciso.
# Nesses casos, é sinalizado um aviso na saída.

print("\n" + "=" * 70)
print("QUALI x QUALI — V de Cramér (corrigido por viés)")
print("=" * 70)

resultados_qlql = []

for i in range(len(variaveis_quali)):
    for j in range(i + 1, len(variaveis_quali)):
        var1 = variaveis_quali[i]
        var2 = variaveis_quali[j]

        # Remove NaN antes de montar a tabela
        sub = df[[var1, var2]].dropna()
        tabela = pd.crosstab(sub[var1], sub[var2])

        # Qui-Quadrado
        chi2, p_valor, gl, esperado = chi2_contingency(tabela)
        n_obs = tabela.sum().sum()

        # Aviso se frequências esperadas forem baixas
        freq_esperada_min = esperado.min()
        aviso = ""
        if freq_esperada_min < 5:
            aviso = (f"⚠ freq. esperada mín={freq_esperada_min:.1f} "
                     f"(<5, p-valor pode ser impreciso)")

        # V de Cramér corrigido (Bergsma, 2013)
        v_cramer = calcular_cramer_corrigido(tabela, chi2, n_obs)

        resultados_qlql.append({
            "Var 1": var1,
            "Var 2": var2,
            "Cramér corr. (V)": round(v_cramer, 4),
            "p-valor χ²": round(p_valor, 4),
            "Classif. Cramér": classificar_cramer(v_cramer),
            "Aviso": aviso,
        })

if resultados_qlql:
    df_qlql = pd.DataFrame(resultados_qlql)
else:
    print("Não há pares de variáveis qualitativas suficientes.")


QUALI x QUALI — V de Cramér (corrigido por viés)


In [18]:
df_qlql.head()

,Var 1,Var 2,Cramér corr. (V),p-valor χ²,Classif. Cramér,Aviso
0,sexo,escolaridade,0.1343,0.3072,Fraca,
1,sexo,regiao,0.0411,0.9501,Desprezível,
2,escolaridade,regiao,0.1040,0.6954,Fraca,


In [19]:
# =============================================================================
# 9. QUALI x QUANTI — Eta e Eta²
# =============================================================================
# Eta (η):   razão de correlação — varia de 0 a 1.
# Eta² (η²): proporção da variância explicada pela variável categórica.
#            Ex: η² = 0.15 → a categórica explica 15% da variação da numérica.
#
# O p-valor vem da ANOVA one-way (teste F). Testa H0: "as médias de todos
# os grupos são iguais". Valores < 0.05 indicam que pelo menos um grupo
# tem média significativamente diferente dos demais.

print("\n" + "=" * 70)
print("QUALI x QUANTI — Eta (η), Eta² (η²)")
print("=" * 70)

resultados_qlqt = []

for var_qual in variaveis_quali:
    for var_quant in variaveis_quanti:

        eta, eta2, p_anova = calcular_eta(df, var_qual, var_quant)

        resultados_qlqt.append({
            "Var. Quali": var_qual,
            "Var. Quanti": var_quant,
            "Eta (η)": round(eta, 4),
            "Classif. Eta": classificar_eta(eta),
            "Eta² (η²)": round(eta2, 4),
            "Var. explicada": f"{round(eta2 * 100, 1)}%",
            "Classif. Eta²": classificar_eta2(eta2),
            "p-valor ANOVA": round(p_anova, 4),
        })

if resultados_qlqt:
    df_qlqt = pd.DataFrame(resultados_qlqt)
else:
    print("Não há pares quali x quanti para calcular.")


QUALI x QUANTI — Eta (η), Eta² (η²)


In [20]:
df_qlqt.head()

,Var. Quali,Var. Quanti,Eta (η),Classif. Eta,Eta² (η²),Var. explicada,Classif. Eta²,p-valor ANOVA
0,sexo,idade,0.1300,Fraca,0.0169,1.7%,Pequeno,0.0666
1,sexo,renda,0.1494,Fraca,0.0223,2.2%,Pequeno,0.0347
2,sexo,nota,0.1301,Fraca,0.0169,1.7%,Pequeno,0.0663
3,escolaridade,idade,0.0974,Desprezível,0.0095,0.9%,Desprezível,0.5991
4,escolaridade,renda,0.1578,Fraca,0.0249,2.5%,Pequeno,0.1751


In [24]:
# =============================================================================
# 11. NOTAS DE INTERPRETAÇÃO
# =============================================================================
# Avisos importantes para evitar os erros mais comuns na interpretação
# de correlações e medidas de associação.

print("\n" + "=" * 70)
print("NOTAS IMPORTANTES DE INTERPRETAÇÃO")
print("=" * 70)

print("""
  1. SIGNIFICÂNCIA ≠ RELEVÂNCIA PRÁTICA
     Em bases grandes, quase toda correlação será significativa
     (p < 0.05), mesmo as muito fracas. Sempre avalie a INTENSIDADE
     (classificação) junto com o p-valor. Uma correlação de 0.03 com
     p < 0.001 é real, mas irrelevante na prática.

  2. CORRELAÇÃO ≠ CAUSALIDADE
     Uma correlação forte entre A e B não prova que A causa B, nem que
     B causa A. Pode haver uma terceira variável (confundidora) que
     influencia ambas. Para inferir causalidade, são necessários
     desenhos experimentais ou técnicas específicas.

  3. VARIÁVEIS CONFUNDIDORAS (CORRELAÇÃO ESPÚRIA)
     Duas variáveis podem parecer correlacionadas apenas porque ambas
     são influenciadas por um terceiro fator. Ex: vendas de sorvete e
     afogamentos se correlacionam porque ambos dependem da temperatura.
     Sempre considere: "existe algo que poderia estar causando ambas?"

  4. PEARSON = 0 NÃO SIGNIFICA "SEM RELAÇÃO"
     O Pearson mede apenas relações LINEARES. Se a relação for curva
     (U, senoidal, exponencial), o Pearson pode dar ~0 mesmo com
     dependência forte. Compare sempre Pearson vs. dCor: se o dCor
     for muito maior que o Pearson, existe relação não-linear.

  5. OUTLIERS PODEM DISTORCER O PEARSON
     Um único ponto extremo pode inflar ou destruir o Pearson. Se
     Pearson e Spearman divergem muito para o mesmo par, investigue
     a presença de outliers. O Spearman é robusto a extremos por
     usar postos em vez de valores brutos.

  6. AMOSTRAS PEQUENAS GERAM CORRELAÇÕES INSTÁVEIS
     Com poucos dados, os coeficientes flutuam muito. Uma correlação
     de 0.40 com n=15 pode virar 0.05 em outra amostra do mesmo
     fenômeno. Desconfie de correlações "fortes" com n pequeno,
     mesmo que o p-valor seja significativo.

  7. CATEGORIAS DESBALANCEADAS REDUZEM O CRAMÉR
     Se uma variável qualitativa é muito desigual (ex: 95% "sim",
     5% "não"), o V de Cramér tende a ser baixo independente da
     relação real. Antes de interpretar, verifique a distribuição
     de frequências das variáveis categóricas.

  8. ETA É ASSIMÉTRICO
     Eta(sexo → renda) mede quanto "sexo" explica a variação de
     "renda". Isso NÃO é o mesmo que perguntar o inverso. O código
     calcula a direção quali → quanti. Se precisar do inverso,
     a pergunta estatística seria diferente.

  9. MÚLTIPLAS COMPARAÇÕES INFLAM FALSOS POSITIVOS
     Com muitas variáveis, o número de pares cresce rápido (ex: 20
     variáveis = 190 pares). Ao nível de 5%, esperamos ~10 pares
     "significativos" POR ACASO, sem nenhuma relação real. Trate
     esta análise como EXPLORATÓRIA: os resultados são hipóteses
     para investigar, não conclusões definitivas.
""")


NOTAS IMPORTANTES DE INTERPRETAÇÃO

  1. SIGNIFICÂNCIA ≠ RELEVÂNCIA PRÁTICA
     Em bases grandes, quase toda correlação será significativa
     (p < 0.05), mesmo as muito fracas. Sempre avalie a INTENSIDADE
     (classificação) junto com o p-valor. Uma correlação de 0.03 com
     p < 0.001 é real, mas irrelevante na prática.

  2. CORRELAÇÃO ≠ CAUSALIDADE
     Uma correlação forte entre A e B não prova que A causa B, nem que
     B causa A. Pode haver uma terceira variável (confundidora) que
     influencia ambas. Para inferir causalidade, são necessários
     desenhos experimentais ou técnicas específicas.

  3. VARIÁVEIS CONFUNDIDORAS (CORRELAÇÃO ESPÚRIA)
     Duas variáveis podem parecer correlacionadas apenas porque ambas
     são influenciadas por um terceiro fator. Ex: vendas de sorvete e
     afogamentos se correlacionam porque ambos dependem da temperatura.
     Sempre considere: "existe algo que poderia estar causando ambas?"

  4. PEARSON = 0 NÃO SIGNIFICA "SEM RELAÇÃO"
   